In [ ]:
import sys
!{sys.executable} -m pip install -U python-woc pandas matplotlib

In [5]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = [
    'Pseudomanifold/Aleph',
    'trichter/rf',
    'evalf/nutils',
    'eXascaleInfolab/bench-vldb20',
    'ashiklom/edr-da',
    'OxfordSKA/OSKAR',
    'ctlab/fgsea',
    'LAMPSPUC/StateSpaceModels.jl',
    'SiEPIC/SiEPIC_EBeam_PDK',
    'PaulHancock/Aegean'
]
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,0026ee632a7cd96e05abdbdc7d1fc765aa815aa6,pseudomanifold_aleph


In [8]:
summary = df_commit_data[
    ['project', 'commit', 'author', 'author_time', 'message']
].copy()

summary.columns = [
    'project_wocid',
    'commit_sha1',
    'author',
    'time',
    'commit message'
]

summary.to_csv(
    'dmengeli_project_summary.csv',
    sep=';',
    index=False
)

summary.head()

,project_wocid,commit_sha1,author,time,commit message
0,pseudomanifold_aleph,0026ee632a7cd96e05abdbdc7d1fc765aa815aa6,Bastian Rieck <bastian.rieck@bsse.ethz.ch>,1519028904,Fixed Python integration for Mac OS X\n\nIt is...
1,pseudomanifold_aleph,002d27c8b90928e024aeeafbd9c32d561e0691b2,Bastian Rieck <bastian.rieck@iwr.uni-heidelber...,1497432549,Fixed functor formatting\n
2,pseudomanifold_aleph,00350cd310715806678cfa26a697079ab8cbb793,Bastian Rieck <bastian@rieck.ru>,1485521337,Fixed step function addition\n\nIt is probably...
3,pseudomanifold_aleph,003910ae73dab404d119e889e30e4245a6a610d0,Bastian Rieck <bastian@rieck.ru>,1578045238,Implemented persistence diagram range insertion\n
4,pseudomanifold_aleph,0088bafb190a245f934392d112a2940ec7d694e4,Bastian Rieck <bastian.rieck@iwr.uni-heidelber...,1486737991,Extended GML reader test\n\nLabels are being p...


In [6]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.item s():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

 11%|███▉                               | 233/2093 [04:00<32:18,  1.04s/it]

Got Errors {'2b14d054203851cac6e049ecf75208121ccdcffe': 'Key 2b14d054203851cac6e049ecf75208121ccdcffe not found in /da5_fast/All.sha1c/commit_43.tch'}


 11%|███▉                               | 234/2093 [04:01<32:18,  1.04s/it]

Got Errors {'31f9ac970a355af0b0a7aa1c81cee5b5135bd22a': 'Key 31f9ac970a355af0b0a7aa1c81cee5b5135bd22a not found in /da5_fast/All.sha1c/commit_49.tch'}


 11%|███▉                               | 235/2093 [04:02<32:23,  1.05s/it]

...

Got Errors {'c7e93c9949c16efaa901aeb71295bc8b9f173dea': 'Key c7e93c9949c16efaa901aeb71295bc8b9f173dea not found in /da5_fast/All.sha1c/commit_71.tch'}


 99%|█████████████████████████████████▋| 2072/2093 [35:50<00:21,  1.04s/it]

Got Errors {'e96da1bb2e3946d87df39f7f47b46509a786e7a7': 'Key e96da1bb2e3946d87df39f7f47b46509a786e7a7 not found in /da5_fast/All.sha1c/commit_105.tch'}


100%|██████████████████████████████████| 2093/2093 [36:11<00:00,  1.04s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,0026ee632a7cd96e05abdbdc7d1fc765aa815aa6,4ca7cb1ff03bcd431873b6e150191cb5e048ebea,[ff4d3decf83f19bec6183365ddc336c44e9bc372],Bastian Rieck <bastian.rieck@bsse.ethz.ch>,1519028904,+0100,Bastian Rieck <bastian.rieck@bsse.ethz.ch>,1519028904,+0100,Fixed Python integration for Mac OS X\n\nIt is...,0026ee632a7cd96e05abdbdc7d1fc765aa815aa6,pseudomanifold_aleph
1,002d27c8b90928e024aeeafbd9c32d561e0691b2,2ffc345f33cc6cc5623feaf9436cd420752c0401,[07d2643da6dfafa71bb6fec111918aca5778dbae],Bastian Rieck <bastian.rieck@iwr.uni-heidelber...,1497432549,+0200,Bastian Rieck <bastian.rieck@iwr.uni-heidelber...,1497432549,+0200,Fixed functor formatting\n,002d27c8b90928e024aeeafbd9c32d561e0691b2,pseudomanifold_aleph


## GitHub Project Statistics

Pseudomanifold/Aleph
Stars: 108
Forks: 15
Last commit date: July 16, 2025

trichter/rf
Stars: 124
Forks: 65
Last commit date: May 5, 2026

evalf/nutils
Stars: 102
Forks: 47
Last commit date: September 21, 2026

eXascaleInfolab/bench-vldb20
Stars: 41
Forks: 15
Last commit date: January 23, 2025

ashiklom/edr-da
Stars: 7
Forks: 4
Last commit date: June 20, 2025

OxfordSKA/OSKAR
Stars: 82
Forks: 46
Last commit date: July 23, 2026

ctlab/fgsea
Stars: 459
Forks: 77
Last commit date: September 22, 2026

LAMPSPUC/StateSpaceModels.jl
Stars: 291
Forks: 30
Last commit date: August 24, 2026

SiEPIC/SiEPIC_EBeam_PDK
Stars: 286
Forks: 184
Last commit date: February 13, 2026

PaulHancock/Aegean
Stars: 52
Forks: 14
Last commit date: July 28, 2025

In [19]:
woc_stats = df_commit_data.groupby('project').agg(
    commits=('commit', 'nunique'),
    authors=('author', 'nunique'),
    max_time=('author_time', 'max'),
    min_time=('author_time', 'min')
)

woc_stats

,commits,authors,max_time,min_time
project,,,,
ashiklom_edr-da,458,5,1750426419,1477337165
ctlab_fgsea,544,33,1762551642,1461616806
evalf_nutils,6587,49,1757000677,1329989907
exascaleinfolab_bench-vldb20,607,10,1737636470,1539276296
lampspuc_statespacemodels.jl,1113,35,1773935433,1516020652
oxfordska_oskar,2743,24,1761294322,1289905814
paulhancock_aegean,2402,23,1753841963,1330396617
pseudomanifold_aleph,2222,10,1752677613,1473339037
siepic_siepic_ebeam_pdk,3658,84,1761227477,1447480391


## World of Code Project Statistics

Project: ashiklom_edr-da
Number of commits: 458
Number of authors: 5
Max time: 1750426419
Min time: 1477337165

Project: ctlab_fgsea
Number of commits: 544
Number of authors: 33
Max time: 1762551642
Min time: 1461616806

Project: evalf_nutils
Number of commits: 6587
Number of authors: 49
Max time: 1757000677
Min time: 1329989907

Project: exascaleinfolab_bench-vldb20
Number of commits: 607
Number of authors: 10
Max time: 1737636470
Min time: 1539276296

Project: lampspuc_statespacemodels.jl
Number of commits: 1113
Number of authors: 35
Max time: 1773935433
Min time: 1516020652

Project: oxfordska_oskar
Number of commits: 2743
Number of authors: 24
Max time: 1761294322
Min time: 1289905814

Project: paulhancock_aegean
Number of commits: 2402
Number of authors: 23
Max time: 1753841963
Min time: 1330396617

Project: pseudomanifold_aleph
Number of commits: 2222
Number of authors: 10
Max time: 1752677613
Min time: 1473339037

Project: siepic_siepic_ebeam_pdk
Number of commits: 3658
Number of authors: 84
Max time: 1761227477
Min time: 1447480391

Project: trichter_rf
Number of commits: 554
Number of authors: 11
Max time: 1771950976
Min time: 1361208896

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github